In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, date_format, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType
import os

kafka_bootstrap_servers = "kafka:9092" #os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:9093")
kafka_topic = os.environ.get("KAFKA_TOPIC", "instagram-post")

# Configurações do S3/MinIO
s3_bucket_name = "warehouse" # Nome do seu bucket no MinIO (criado pelo `mc` do docker-compose)
output_path = f"s3a://{s3_bucket_name}/bronze/instagram/" # Caminho de saída no MinIO


In [2]:
print(kafka_bootstrap_servers, kafka_topic)

kafka:9092 instagram-post


In [3]:
from py4j.java_gateway import JavaObject

jars = spark.sparkContext._jsc.sc().listJars()
jars_list = [jars.apply(i) for i in range(jars.size())]
for jar in jars_list:
    print(jar)

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IcebergS3") \
    .config("spark.jars", "/opt/spark/jars/*")\
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.extraJavaOptions", "-verbose:class") \
    .config("spark.executor.extraJavaOptions", "-verbose:class") \
    .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "random") \
    .getOrCreate()

25/07/29 02:32:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:


# Leitura do tópico Kafka
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()




In [6]:
comment_schema = StructType([
    StructField("user", StringType(), True),
    StructField("comment", StringType(), True),
    StructField("timestamp", StringType(), True)
])
# Schema do JSON enviado
instagram_schema = StructType([
    StructField("id", StringType(), True),
    StructField("user_handle", StringType(), True),
    StructField("caption", StringType(), True),
    StructField("image_url", StringType(), True),
    StructField("posted_at", StringType(), True),  # ou TimestampType se já estiver parseado
    StructField("likes", IntegerType(), True),
    StructField("hashtags", ArrayType(StringType()), True),
    StructField("comments", ArrayType(comment_schema), True)
])

In [7]:

# Extração e parsing do JSON
df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json") \
    .withColumn("data", from_json(col("json"), instagram_schema)) \
    .select("data.*")
df_parsed = df_parsed.withColumn("event_time", current_timestamp())\
    .withColumn("ingestion_date", date_format(col("event_time"), "yyyy-MM-dd"))

# Escrever os dados brutos na camada Bronze do Data Lake
query = (df_parsed.coalesce(1).writeStream\
    .format("parquet")\
    .partitionBy("ingestion_date")\
    .option("path", output_path)\
    .option("checkpointLocation", f"s3a://{s3_bucket_name}/_spark_checkpoint/instagram/") \
    .trigger(processingTime="10 minutes")\
    .outputMode("append")\
    .start())


Py4JJavaError: An error occurred while calling o83.start.
: java.lang.NoClassDefFoundError: org/apache/hadoop/fs/impl/prefetch/PrefetchingStatistics
	at java.base/java.lang.ClassLoader.defineClass1(Native Method)
	at java.base/java.lang.ClassLoader.defineClass(ClassLoader.java:1017)
	at java.base/java.security.SecureClassLoader.defineClass(SecureClassLoader.java:150)
	at java.base/jdk.internal.loader.BuiltinClassLoader.defineClass(BuiltinClassLoader.java:862)
	at java.base/jdk.internal.loader.BuiltinClassLoader.findClassOnClassPathOrNull(BuiltinClassLoader.java:760)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClassOrNull(BuiltinClassLoader.java:681)
	at java.base/jdk.internal.loader.BuiltinClassLoader.loadClass(BuiltinClassLoader.java:639)
	at java.base/jdk.internal.loader.ClassLoaders$AppClassLoader.loadClass(ClassLoaders.java:188)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.initialize(S3AFileSystem.java:5

In [ ]:

query.awaitTermination()